# Домашнее задание №8. Трансформеры для RuCoLA

**Задача:** бинарная классификация лингвистической приемлемости русских предложений (`1` — приемлемо, `0` — неприемлемо).

В ноутбуке выполнены все пункты задания:

1. `in_domain_train.csv` стратифицированно разделён на train/validation, `in_domain_dev.csv` оставлен как независимый test.
2. Fine-tuning **RuBERT-large** как классификатора.
3. **RuGPT-3 large** в zero-/few-shot режиме: 3 формулировки промпта и 0, 1, 2, 4 демонстрации.
4. Fine-tuning **RuT5-base** в постановке text-to-text.
5. Единые метрики Accuracy, F1 и Matthews correlation coefficient (MCC), анализ ошибок и ответы на вопросы.

> В исходном задании идентификатор `sberbank-ai/ruBert-larg` содержит опечатку. Использован актуальный идентификатор `ai-forever/ruBert-large` (прежнее имя организации `sberbank-ai` перенаправляется на `ai-forever`). Основная метрика — MCC: она информативнее Accuracy при дисбалансе классов.

## 0. Окружение

Полный прогон рассчитан на GPU (рекомендуется Google Colab с GPU уровня T4/A100). Модели скачиваются из Hugging Face при первом запуске.

- `RUN_MODE = 'teacher'` — быстрая проверка всех веток на небольших русскоязычных checkpoints и подвыборках; подходит преподавателю для `Run All`, но его метрики не являются отчётными.
- `RUN_MODE = 'full'` — модели и полный объём данных строго по заданию; именно этот режим используется для получения итоговых чисел.

In [ ]:
%pip install -q "torch>=2.2,<3" "transformers>=4.45,<5" "datasets>=3,<4" "accelerate>=1,<2" "sentencepiece>=0.2,<1" "scikit-learn>=1.4,<2" "pandas>=2,<3" "matplotlib>=3.8,<4" "seaborn>=0.13,<1" "tabulate>=0.9,<1"

In [ ]:
import gc
import inspect
import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from IPython.display import Markdown, display
from matplotlib import pyplot as plt
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, matthews_corrcoef,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM, AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
    DataCollatorWithPadding, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    Trainer, TrainingArguments, set_seed,
)

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
RUN_MODE = 'teacher'  # 'teacher' — быстрая проверка; 'full' — итоговый эксперимент
assert RUN_MODE in {'teacher', 'full'}
FAST_RUN = RUN_MODE == 'teacher'
VAL_SIZE = 0.20
MAX_LENGTH = 64 if FAST_RUN else 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if RUN_MODE == 'full':
    BERT_MODEL_NAME = 'ai-forever/ruBert-large'
    GPT_MODEL_NAME = 'ai-forever/rugpt3large_based_on_gpt2'
    T5_MODEL_NAME = 'ai-forever/ruT5-base'
    BERT_LABEL, GPT_LABEL, T5_LABEL = 'RuBERT-large', 'RuGPT-3 large', 'RuT5-base'
else:
    # Компактные русскоязычные checkpoints используются только для проверки кода.
    BERT_MODEL_NAME = 'ai-forever/ruBert-base'
    GPT_MODEL_NAME = 'ai-forever/rugpt3small_based_on_gpt2'
    T5_MODEL_NAME = 'cointegrated/rut5-small'
    BERT_LABEL, GPT_LABEL, T5_LABEL = (
        'RuBERT-base [teacher check]',
        'RuGPT-3 small [teacher check]',
        'RuT5-small [teacher check]',
    )

ARTIFACTS = Path('artifacts')
ARTIFACTS.mkdir(exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

seed_everything()
print(f'mode={RUN_MODE}; device={DEVICE}; torch={torch.__version__}; CUDA={torch.cuda.is_available()}')
print('models:', BERT_MODEL_NAME, GPT_MODEL_NAME, T5_MODEL_NAME, sep='\n  - ')
if DEVICE.type == 'cpu':
    print('ВНИМАНИЕ: режим full на CPU очень медленный. Для отчётного запуска включите GPU.')

## 1. Данные и протокол эксперимента

`in_domain_train.csv` делится на train/validation с сохранением долей классов. Внешний `in_domain_dev.csv` по условию считается тестом. На validation выбираются checkpoint RuBERT/RuT5 и конфигурация RuGPT; test не участвует в выборе гиперпараметров.

In [ ]:
def find_data_file(name):
    candidates = [Path(name), Path.cwd() / name, Path('/content') / name]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Не найден {name}. Положите CSV рядом с ноутбуком или в /content.')

train_path = find_data_file('in_domain_train.csv')
test_path = find_data_file('in_domain_dev.csv')
raw_train = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

required = {'id', 'sentence', 'acceptable', 'error_type', 'detailed_source'}
for name, frame in [('train', raw_train), ('test', test_df)]:
    assert required.issubset(frame.columns), f'{name}: отсутствуют колонки {required - set(frame.columns)}'
    assert frame['sentence'].notna().all(), f'{name}: есть пропуски в sentence'
    assert set(frame['acceptable'].unique()) == {0, 1}, f'{name}: метка должна быть 0/1'
    assert not frame['id'].duplicated().any(), f'{name}: id должны быть уникальны'

print('train source:', train_path.resolve())
print('test source: ', test_path.resolve())
display(raw_train.head())

In [ ]:
summary = pd.DataFrame({
    'split': ['in_domain_train', 'in_domain_dev (test)'],
    'n': [len(raw_train), len(test_df)],
    'acceptable_1': [raw_train.acceptable.sum(), test_df.acceptable.sum()],
    'unacceptable_0': [(raw_train.acceptable == 0).sum(), (test_df.acceptable == 0).sum()],
    'positive_share': [raw_train.acceptable.mean(), test_df.acceptable.mean()],
})
display(summary.style.format({'positive_share': '{:.2%}'}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=raw_train, x='acceptable', ax=axes[0])
axes[0].set(title='Классы в исходном train', xlabel='acceptable')
(raw_train.error_type.value_counts().drop('0', errors='ignore')
 .plot.bar(ax=axes[1], title='Типы ошибок в train'))
axes[1].set(xlabel='error_type', ylabel='count')
plt.tight_layout()
plt.show()

print('Наблюдение: класс 1 преобладает, поэтому наряду с Accuracy используем F1 и MCC.')

In [ ]:
train_df, val_df = train_test_split(
    raw_train, test_size=VAL_SIZE, random_state=SEED, stratify=raw_train['acceptable']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

def stratified_limit(df, n):
    if n is None or n >= len(df):
        return df.copy().reset_index(drop=True)
    selected, _ = train_test_split(
        df, train_size=n, random_state=SEED, stratify=df['acceptable']
    )
    return selected.reset_index(drop=True)

train_work = stratified_limit(train_df, 64 if FAST_RUN else None)
val_work = stratified_limit(val_df, 24 if FAST_RUN else None)
test_work = stratified_limit(test_df, 32 if FAST_RUN else None)

splits = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_df), len(val_df), len(test_df)],
    'used_now': [len(train_work), len(val_work), len(test_work)],
    'acceptable_share': [train_df.acceptable.mean(), val_df.acceptable.mean(), test_df.acceptable.mean()],
})
display(splits.style.format({'acceptable_share': '{:.2%}'}))
assert set(train_df.id).isdisjoint(val_df.id)
print('Утечки между train и validation нет.')

In [ ]:
def metrics_dict(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }

def compute_metrics_from_logits(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    return metrics_dict(labels, np.argmax(logits, axis=-1))

def show_evaluation(name, y_true, y_pred):
    m = metrics_dict(y_true, y_pred)
    print(name, {k: round(v, 4) for k, v in m.items()})
    print(classification_report(y_true, y_pred, target_names=['неприемлемо', 'приемлемо'], digits=4))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Предсказание')
    plt.ylabel('Истинная метка')
    plt.title(name)
    plt.show()
    return m

results = []
test_predictions = {}
majority_label = int(train_work.acceptable.mode().iloc[0])
majority_pred = np.full(len(test_work), majority_label)
majority_metrics = metrics_dict(test_work.acceptable, majority_pred)
results.append({'model': 'Majority baseline', **majority_metrics})
print('Majority baseline:', majority_metrics)

## 2. Fine-tuning RuBERT

Добавляется линейная голова на два класса. Сообщение `classifier.weight/bias were not initialized` **ожидаемо**: базовый checkpoint предобучен без головы классификации, поэтому новая голова инициализируется случайно и обучается в следующей ячейке. Padding выполняется динамически внутри batch. Лучший checkpoint выбирается по validation MCC. В режиме `full` используется RuBERT-large и 3 эпохи; в режиме `teacher` — RuBERT-base, 64 примера и 1 эпоха.

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

def to_hf_dataset(df, tokenizer, seq2seq=False):
    ds = Dataset.from_pandas(df[['sentence', 'acceptable']], preserve_index=False)
    def tokenize_batch(batch):
        encoded = tokenizer(
            batch['sentence'], truncation=True, max_length=MAX_LENGTH, padding=False
        )
        encoded['labels'] = batch['acceptable']
        return encoded
    return ds.map(tokenize_batch, batched=True, remove_columns=ds.column_names)

bert_train_ds = to_hf_dataset(train_work, bert_tokenizer)
bert_val_ds = to_hf_dataset(val_work, bert_tokenizer)
bert_test_ds = to_hf_dataset(test_work, bert_tokenizer)
bert_collator = DataCollatorWithPadding(bert_tokenizer, pad_to_multiple_of=8 if DEVICE.type == 'cuda' else None)
print(bert_train_ds)

In [ ]:
seed_everything()
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=2,
    id2label={0: 'unacceptable', 1: 'acceptable'},
    label2id={'unacceptable': 0, 'acceptable': 1},
)
print('Новая classifier-голова — ожидаемо; сейчас она будет обучена на RuCoLA.')

bert_args_kwargs = dict(
    output_dir=str(ARTIFACTS / 'rubert_checkpoints'),
    learning_rate=2e-5, weight_decay=0.01,
    per_device_train_batch_size=4 if DEVICE.type == 'cuda' else 2,
    per_device_eval_batch_size=16 if DEVICE.type == 'cuda' else 4,
    gradient_accumulation_steps=4,
    num_train_epochs=1 if FAST_RUN else 3,
    warmup_ratio=0.1, lr_scheduler_type='linear',
    save_strategy='epoch', logging_steps=10 if FAST_RUN else 50,
    load_best_model_at_end=True, metric_for_best_model='mcc', greater_is_better=True,
    save_total_limit=1, fp16=(DEVICE.type == 'cuda'),
    dataloader_pin_memory=(DEVICE.type == 'cuda'),
    report_to='none', seed=SEED, data_seed=SEED,
)
strategy_name = 'eval_strategy' if 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters else 'evaluation_strategy'
bert_args_kwargs[strategy_name] = 'epoch'
bert_args = TrainingArguments(**bert_args_kwargs)

bert_trainer = Trainer(
    model=bert_model, args=bert_args,
    train_dataset=bert_train_ds, eval_dataset=bert_val_ds,
    data_collator=bert_collator, compute_metrics=compute_metrics_from_logits,
)
start = time.time()
bert_trainer.train()
print(f'RuBERT training: {(time.time() - start) / 60:.1f} min')
bert_trainer.save_model(str(ARTIFACTS / 'rubert-best'))
bert_tokenizer.save_pretrained(str(ARTIFACTS / 'rubert-best'))

In [ ]:
bert_output = bert_trainer.predict(bert_test_ds)
bert_pred = np.argmax(bert_output.predictions, axis=-1)
bert_metrics = show_evaluation(f'{BERT_LABEL} / test', test_work.acceptable.to_numpy(), bert_pred)
results.append({'model': f'{BERT_LABEL} fine-tuning', **bert_metrics})
test_predictions[f'{BERT_LABEL} fine-tuning'] = bert_pred
pd.DataFrame({'id': test_work.id, 'prediction': bert_pred}).to_csv(
    ARTIFACTS / 'rubert_test_predictions.csv', index=False
)

# Освобождаем GPU перед RuGPT-3.
del bert_output, bert_trainer, bert_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 3. RuGPT-3 large: zero-shot и few-shot

Проверяются 3 формулировки инструкции и `k ∈ {0, 1, 2, 4}` демонстраций — всего 12 конфигураций. Демонстрации берутся только из train в фиксированном порядке классов `1, 0, 1, 0`.

Вместо нестабильного свободного `generate()` используется стандартное **verbalizer scoring**: для каждого промпта сравнивается средняя условная log-probability продолжений «допустимо» и «недопустимо». Это по-прежнему zero/few-shot, не требует обучения RuGPT и делает решение детерминированным. Все 12 конфигураций сравниваются на validation; победитель один раз оценивается на test.

In [ ]:
seed_everything()
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL_NAME)
if gpt_tokenizer.pad_token_id is None:
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
gpt_tokenizer.padding_side = 'right'
gpt_tokenizer.truncation_side = 'left'

gpt_dtype = torch.float16 if DEVICE.type == 'cuda' else torch.float32
gpt_model = AutoModelForCausalLM.from_pretrained(
    GPT_MODEL_NAME, torch_dtype=gpt_dtype, low_cpu_mem_usage=True
).to(DEVICE)
gpt_model.eval()
max_context = int(getattr(gpt_model.config, 'n_positions', 2048))
print(f'{GPT_MODEL_NAME}: parameters={sum(p.numel() for p in gpt_model.parameters()):,}, context={max_context}')

In [ ]:
PROMPT_TEMPLATES = {
    'short': (
        'Определи грамматическую приемлемость русского предложения.\n'
        '{examples}Предложение: {sentence}\nОтвет:'
    ),
    'expert': (
        'Ты — лингвист. Класс «допустимо» означает естественное и грамматически корректное предложение; '
        '«недопустимо» — предложение с морфологической, синтаксической или семантической ошибкой.\n'
        '{examples}Предложение: {sentence}\nКласс:'
    ),
    'binary': (
        'Задача бинарной классификации русских предложений. Возможные ответы: допустимо или недопустимо. '
        'Верни только один ответ.\n{examples}Текст: {sentence}\nМетка:'
    ),
}
VERBALIZERS = {0: ' недопустимо', 1: ' допустимо'}

# Фиксированная последовательность классов; примеры не пересекаются с validation/test.
demo_rows = []
for label in [1, 0, 1, 0]:
    used_ids = {row['id'] for row in demo_rows}
    row = train_df[(train_df.acceptable == label) & (~train_df.id.isin(used_ids))].sample(
        1, random_state=SEED + len(demo_rows)
    ).iloc[0]
    demo_rows.append(row.to_dict())

def demo_text(k):
    return ''.join(
        f"Предложение: {row['sentence']}\nОтвет: {VERBALIZERS[row['acceptable']].strip()}\n\n"
        for row in demo_rows[:k]
    )

def make_prompts(sentences, template_name, k):
    template = PROMPT_TEMPLATES[template_name]
    examples = demo_text(k)
    return [template.format(examples=examples, sentence=s) for s in sentences]

display(pd.DataFrame(demo_rows)[['sentence', 'acceptable']])
print(make_prompts([val_work.sentence.iloc[0]], 'expert', 2)[0])

In [ ]:
@torch.inference_mode()
def score_verbalizer(prompts, verbalizer, batch_size=None):
    """Средняя log P(токены verbalizer | prompt), с учётом causal shift."""
    if batch_size is None:
        batch_size = 8 if DEVICE.type == 'cuda' else 2
    suffix_ids = gpt_tokenizer.encode(verbalizer, add_special_tokens=False)
    encoded = []
    for prompt in prompts:
        prefix_ids = gpt_tokenizer.encode(
            prompt, add_special_tokens=False, truncation=True,
            max_length=max_context - len(suffix_ids)
        )
        encoded.append(prefix_ids + suffix_ids)

    scores = []
    for start_idx in range(0, len(encoded), batch_size):
        chunk = encoded[start_idx:start_idx + batch_size]
        lengths = [len(x) for x in chunk]
        max_len = max(lengths)
        input_ids = torch.full(
            (len(chunk), max_len), gpt_tokenizer.pad_token_id, dtype=torch.long, device=DEVICE
        )
        attention_mask = torch.zeros_like(input_ids)
        for i, ids in enumerate(chunk):
            input_ids[i, :len(ids)] = torch.tensor(ids, device=DEVICE)
            attention_mask[i, :len(ids)] = 1
        logits = gpt_model(input_ids=input_ids, attention_mask=attention_mask).logits
        log_probs = torch.log_softmax(logits.float(), dim=-1)
        for i, total_len in enumerate(lengths):
            suffix_start = total_len - len(suffix_ids)
            token_positions = torch.arange(suffix_start, total_len, device=DEVICE)
            # Токен на позиции t предсказывается логитами позиции t-1.
            lp = log_probs[i, token_positions - 1, input_ids[i, token_positions]]
            scores.append(lp.mean().item())
    return np.asarray(scores)

def gpt_predict(df, template_name, k):
    prompts = make_prompts(df.sentence.tolist(), template_name, k)
    score_0 = score_verbalizer(prompts, VERBALIZERS[0])
    score_1 = score_verbalizer(prompts, VERBALIZERS[1])
    pred = (score_1 > score_0).astype(int)
    return pred, np.column_stack([score_0, score_1])

In [ ]:
gpt_rows = []
for template_name in PROMPT_TEMPLATES:
    for k in [0, 1, 2, 4]:
        started = time.time()
        pred, scores = gpt_predict(val_work, template_name, k)
        metrics = metrics_dict(val_work.acceptable, pred)
        gpt_rows.append({'template': template_name, 'shots': k, **metrics})
        print(template_name, k, {x: round(v, 4) for x, v in metrics.items()},
              f'{time.time() - started:.1f}s')

gpt_search = pd.DataFrame(gpt_rows).sort_values(['mcc', 'f1'], ascending=False).reset_index(drop=True)
display(gpt_search.style.format({'accuracy': '{:.4f}', 'f1': '{:.4f}', 'mcc': '{:.4f}'}))
gpt_search.to_csv(ARTIFACTS / 'gpt_prompt_search_validation.csv', index=False)

best_gpt = gpt_search.iloc[0]
best_template, best_k = best_gpt['template'], int(best_gpt['shots'])
print(f'Выбрано только по validation: template={best_template}, shots={best_k}')

gpt_test_pred, gpt_test_scores = gpt_predict(test_work, best_template, best_k)
gpt_metrics = show_evaluation(
    f'{GPT_LABEL} ({best_template}, {best_k}-shot) / test',
    test_work.acceptable.to_numpy(), gpt_test_pred
)
results.append({'model': f'{GPT_LABEL} {best_k}-shot', **gpt_metrics})
test_predictions[f'{GPT_LABEL} {best_k}-shot'] = gpt_test_pred
pd.DataFrame({
    'id': test_work.id, 'score_0': gpt_test_scores[:, 0],
    'score_1': gpt_test_scores[:, 1], 'prediction': gpt_test_pred
}).to_csv(ARTIFACTS / 'rugpt3_test_predictions.csv', index=False)

In [ ]:
# Ответы на пункты 3a и 3b визуализируются по validation, не по test.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.lineplot(data=gpt_search, x='shots', y='mcc', hue='template', marker='o', ax=axes[0])
axes[0].set(title='3a/3b: MCC для промптов и числа демонстраций', xticks=[0, 1, 2, 4])
shot_summary = gpt_search.groupby('shots')[['accuracy', 'f1', 'mcc']].agg(['mean', 'std', 'max'])
display(shot_summary.style.format('{:.4f}'))
spread = gpt_search.groupby('shots').mcc.agg(['min', 'max'])
spread['range'] = spread['max'] - spread['min']
spread.plot.bar(y='range', ax=axes[1], legend=False, color='darkorange')
axes[1].set(title='Чувствительность к формулировке: range MCC', ylabel='max − min')
plt.tight_layout()
plt.show()

# Освобождаем GPU перед RuT5.
del gpt_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Fine-tuning RuT5-base

Классификация формулируется как генерация метки: вход `rucola: <предложение>`, целевой текст — `0` или `1`. На validation и test используется `generate`, поэтому оценка соответствует реальному способу применения seq2seq-модели. Лучший checkpoint выбирается по validation MCC.

In [ ]:
seed_everything()
t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL_NAME)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL_NAME)

def make_t5_dataset(df):
    ds = Dataset.from_pandas(df[['sentence', 'acceptable']], preserve_index=False)
    def preprocess(batch):
        model_inputs = t5_tokenizer(
            ['rucola: ' + s for s in batch['sentence']],
            max_length=MAX_LENGTH, truncation=True, padding=False
        )
        labels = t5_tokenizer(
            text_target=[str(x) for x in batch['acceptable']],
            max_length=4, truncation=True, padding=False
        )
        model_inputs['labels'] = labels['input_ids']
        return model_inputs
    return ds.map(preprocess, batched=True, remove_columns=ds.column_names)

t5_train_ds = make_t5_dataset(train_work)
t5_val_ds = make_t5_dataset(val_work)
t5_test_ds = make_t5_dataset(test_work)
t5_collator = DataCollatorForSeq2Seq(t5_tokenizer, model=t5_model, pad_to_multiple_of=8 if DEVICE.type == 'cuda' else None)

def parse_t5_labels(token_ids):
    token_ids = np.where(np.asarray(token_ids) == -100, t5_tokenizer.pad_token_id, token_ids)
    texts = t5_tokenizer.batch_decode(token_ids, skip_special_tokens=True)
    # Любой нераспознанный вывод считаем классом 0 и отдельно контролируем долю таких случаев.
    parsed = np.array([1 if text.strip().startswith('1') else 0 for text in texts])
    valid = np.array([text.strip().startswith(('0', '1')) for text in texts])
    return parsed, valid, texts

def t5_compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    pred, valid, _ = parse_t5_labels(predictions)
    true, _, _ = parse_t5_labels(labels)
    return {**metrics_dict(true, pred), 'valid_output_rate': valid.mean()}

In [ ]:
t5_args_kwargs = dict(
    output_dir=str(ARTIFACTS / 'rut5_checkpoints'),
    learning_rate=3e-4, weight_decay=0.01,
    per_device_train_batch_size=8 if DEVICE.type == 'cuda' else 2,
    per_device_eval_batch_size=16 if DEVICE.type == 'cuda' else 4,
    gradient_accumulation_steps=2,
    num_train_epochs=1 if FAST_RUN else 3,
    warmup_ratio=0.1, lr_scheduler_type='linear',
    save_strategy='epoch', logging_steps=10 if FAST_RUN else 50,
    load_best_model_at_end=True, metric_for_best_model='mcc', greater_is_better=True,
    save_total_limit=1, fp16=(DEVICE.type == 'cuda'),
    dataloader_pin_memory=(DEVICE.type == 'cuda'),
    predict_with_generate=True, generation_max_length=4, generation_num_beams=1,
    report_to='none', seed=SEED, data_seed=SEED,
)
strategy_name = 'eval_strategy' if 'eval_strategy' in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters else 'evaluation_strategy'
t5_args_kwargs[strategy_name] = 'epoch'
t5_args = Seq2SeqTrainingArguments(**t5_args_kwargs)

t5_trainer = Seq2SeqTrainer(
    model=t5_model, args=t5_args,
    train_dataset=t5_train_ds, eval_dataset=t5_val_ds,
    data_collator=t5_collator, compute_metrics=t5_compute_metrics,
)
start = time.time()
t5_trainer.train()
print(f'RuT5 training: {(time.time() - start) / 60:.1f} min')
t5_trainer.save_model(str(ARTIFACTS / 'rut5-best'))
t5_tokenizer.save_pretrained(str(ARTIFACTS / 'rut5-best'))

In [ ]:
t5_output = t5_trainer.predict(t5_test_ds)
t5_pred, t5_valid, t5_texts = parse_t5_labels(t5_output.predictions)
print(f'Корректно распознанные ответы T5: {t5_valid.mean():.2%}')
if not t5_valid.all():
    print('Примеры нераспознанных генераций:', sorted(set(np.asarray(t5_texts)[~t5_valid]))[:10])
t5_metrics = show_evaluation(f'{T5_LABEL} / test', test_work.acceptable.to_numpy(), t5_pred)
results.append({'model': f'{T5_LABEL} fine-tuning', **t5_metrics})
test_predictions[f'{T5_LABEL} fine-tuning'] = t5_pred
pd.DataFrame({'id': test_work.id, 'generated': t5_texts, 'prediction': t5_pred}).to_csv(
    ARTIFACTS / 'rut5_test_predictions.csv', index=False
)

## 5. Итоговое сравнение и анализ ошибок

Все строки ниже относятся к одному и тому же независимому test. Baseline показывает, почему одной Accuracy недостаточно: предсказание преобладающего класса может иметь высокую Accuracy, но MCC равен нулю.

In [ ]:
comparison = pd.DataFrame(results).sort_values('mcc', ascending=False).reset_index(drop=True)
display(comparison.style.format({'accuracy': '{:.4f}', 'f1': '{:.4f}', 'mcc': '{:.4f}'})
        .background_gradient(subset=['mcc'], cmap='Greens'))
comparison.to_csv(ARTIFACTS / 'model_comparison_test.csv', index=False)

long_metrics = comparison.melt(id_vars='model', var_name='metric', value_name='value')
plt.figure(figsize=(12, 5))
sns.barplot(data=long_metrics, x='model', y='value', hue='metric')
plt.axhline(0, color='black', linewidth=.8)
plt.xticks(rotation=15, ha='right')
plt.ylim(min(-0.05, long_metrics.value.min() - 0.05), 1.0)
plt.title('Сравнение моделей на in_domain_dev (test)')
plt.tight_layout()
plt.show()

In [ ]:
def slice_analysis(df, predictions):
    rows = []
    for model_name, pred in predictions.items():
        temp = df[['acceptable', 'error_type']].copy()
        temp['pred'] = pred
        # Для ошибочных предложений accuracy среза = recall обнаружения ошибки (предсказан 0).
        for error_type, part in temp.groupby('error_type'):
            rows.append({
                'model': model_name, 'slice': str(error_type), 'n': len(part),
                'slice_accuracy': accuracy_score(part.acceptable, part.pred),
            })
    return pd.DataFrame(rows)

slices = slice_analysis(test_work, test_predictions)
display(slices.pivot(index='slice', columns='model', values='slice_accuracy')
        .style.format('{:.3f}').background_gradient(axis=1, cmap='RdYlGn'))
slices.to_csv(ARTIFACTS / 'error_type_analysis_test.csv', index=False)

# Самые уверенные ошибки RuGPT полезны для качественного разбора.
gpt_errors = test_work[['id', 'sentence', 'acceptable', 'error_type']].copy()
gpt_errors['prediction'] = gpt_test_pred
gpt_errors['margin'] = np.abs(gpt_test_scores[:, 1] - gpt_test_scores[:, 0])
display(gpt_errors[gpt_errors.acceptable != gpt_errors.prediction]
        .sort_values('margin', ascending=False).head(15))

## 6. Ответы на вопросы и выводы

Следующая ячейка формирует ответы из реально полученных метрик, поэтому текст не содержит выдуманных чисел и автоматически обновляется после полного или быстрого прогона.

In [ ]:
best_model = comparison.iloc[0]
best_prompt_rows = gpt_search.loc[gpt_search.groupby('shots').mcc.idxmax()].sort_values('shots')
shot_means = gpt_search.groupby('shots').mcc.mean()
best_mean_k = int(shot_means.idxmax())
zero_mean = float(shot_means.loc[0])
best_mean = float(shot_means.loc[best_mean_k])
prompt_range_max = float(spread['range'].max())
monotonic = bool(np.all(np.diff(shot_means.sort_index().values) >= -1e-12))

answer = f"""
### 6.1. Варианты затравок (пункт а)

Проверены три варианта: короткая инструкция (`short`), подробная роль лингвиста с определениями классов (`expert`) и формулировка бинарной классификации (`binary`). Максимальный размах MCC между формулировками при фиксированном числе примеров составил **{prompt_range_max:.4f}**. Следовательно, RuGPT-3 {'существенно' if prompt_range_max >= 0.05 else 'умеренно'} чувствителен к формулировке. Лучшие варианты для каждого `k`:

{best_prompt_rows[['shots', 'template', 'mcc']].to_markdown(index=False)}

На validation выбрана конфигурация **{best_template}, {best_k}-shot**; только она применена к test.

### 6.2. Число few-shot примеров (пункт б)

Проверены **0, 1, 2 и 4** примера. Лучшее среднее по трём промптам получено при `k={best_mean_k}`: MCC изменился с **{zero_mean:.4f}** в zero-shot до **{best_mean:.4f}**. Зависимость {'монотонна' if monotonic else 'не монотонна'}: дополнительные демонстрации не гарантируют улучшение, поскольку результат зависит от их состава, порядка и согласования с инструкцией. Для надёжной оценки стоило бы повторить выбор демонстраций с несколькими seed.

### 6.3. Сравнение подходов

Лучший результат на независимом test показал **{best_model['model']}**: Accuracy={best_model['accuracy']:.4f}, F1={best_model['f1']:.4f}, MCC={best_model['mcc']:.4f}. Fine-tuned модели обучаются непосредственно на RuCoLA, тогда как RuGPT-3 решает новую задачу без обновления весов; поэтому few-shot дешевле в обучении, но обычно чувствительнее к промпту и демонстрациям. MCC является главным критерием вывода из-за дисбаланса классов.

### 6.4. Ограничения

Результаты относятся к одному split и одному seed. `in_domain_dev` нельзя использовать для подбора промпта или checkpoint. Для строгого исследования нужны несколько seed fine-tuning, доверительные интервалы, перебор learning rate и повторный few-shot с разными наборами/порядками демонстраций. В `FAST_RUN=True` числа являются только smoke-test и не должны выдаваться за итоговые.
"""
display(Markdown(answer))

run_metadata = {
    'seed': SEED, 'fast_run': FAST_RUN, 'device': str(DEVICE),
    'bert_model': BERT_MODEL_NAME, 'gpt_model': GPT_MODEL_NAME, 't5_model': T5_MODEL_NAME,
    'train_rows': len(train_work), 'validation_rows': len(val_work), 'test_rows': len(test_work),
    'best_gpt_template': str(best_template), 'best_gpt_shots': best_k,
}
with open(ARTIFACTS / 'run_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2)
print('Артефакты сохранены в', ARTIFACTS.resolve())

## 7. Воспроизводимость и источники

- Seed зафиксирован во всех генераторах случайности и `Trainer`.
- Разделение стратифицировано; test не используется при выборе модели.
- Сохраняются лучшие checkpoints, тестовые предсказания, таблицы метрик и параметры запуска.
- Репозиторий и baseline RuCoLA: https://github.com/RussianNLP/RuCoLA
- RuBERT-large: https://huggingface.co/ai-forever/ruBert-large
- RuGPT-3 large: https://huggingface.co/ai-forever/rugpt3large_based_on_gpt2
- RuT5-base: https://huggingface.co/ai-forever/ruT5-base
- Статья RuCoLA: https://aclanthology.org/2022.emnlp-main.348

**Итоговый запуск:** установить `RUN_MODE='full'`, выбрать GPU и выполнить `Restart & Run All`. **Проверка преподавателем:** оставить `RUN_MODE='teacher'` и выполнить `Run All`; пройдут те же этапы, но на компактных моделях и подвыборках.